**Step 1 - Install Libraries**

In [1]:
!pip install -q pypdf faiss-cpu sentence-transformers google-genai

**Step 2 - Imports**

In [1]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google import genai

import faiss
import numpy as np

**Step 3 - Gemini API Key**

In [15]:
GEMINI_API_KEY = "AQ.Ab8RN6JnIK8jkGqYesGdUATgk5LQDDuQaOVE-s42ztDF9jBpvw"

client = genai.Client(
    api_key=GEMINI_API_KEY
)

print("Gemini Connected")

Gemini Connected


**Step 4 - Select PDF**

In [3]:
pdf_path = input("Enter PDF Path: ").strip()

print("Selected:", pdf_path)

Enter PDF Path:  D:\MCA\AIML Training\Python\Complete Guide of Pandas1.pdf


Selected: D:\MCA\AIML Training\Python\Complete Guide of Pandas1.pdf


**Step 5 - Read PDF**

In [4]:
reader = PdfReader(pdf_path)

text = ""

for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
        text += page_text

print("Characters:", len(text))
print("\nPreview:\n")
print(text[:1000])

Characters: 7537

Preview:

Complete Guide of Pandas : By Aniket Sharma 
 
1. What is Pandas in Python, and why is it used? 
Question: What is the primary purpose of the Pandas library in 
Python, and what are its core data structures? 
Answer: 
Pandas is an open-source Python library used for data manipulation 
and analysis. It provides easy-to-use data structures and tools for 
handling structured data, such as tabular data in spreadsheets or SQL 
tables. 
Core Data Structures: 
 Series: A one-dimensional labeled array capable of holding any 
data type. 
 DataFrame: A two-dimensional labeled data structure with 
rows and columns, similar to a spreadsheet or SQL table. 
Pandas is widely used for tasks like data cleaning, transformation, 
aggregation, and merging. 2. Explain the difference between a Pandas Series and 
a DataFrame. 
Question: How do Series and DataFrames differ in Pandas? 
Series: 
 One-dimensional labeled array. 
 Homogeneous data (all elements are of the same type

**Step 6 - Create Chunks**

In [5]:
chunk_size = 500

chunks = []

for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i + chunk_size])

print("Total Chunks:", len(chunks))

Total Chunks: 16


**Step 7 - Load Embedding Model**

In [6]:
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding Model Loaded


**Step 8 - Create Embeddings**

In [7]:
embeddings = embedding_model.encode(
    chunks,
    show_progress_bar=True
)

print("Shape:", embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Shape: (16, 384)


**Step 9 - Create FAISS Database**

In [8]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(embeddings).astype("float32")
)

print("Stored Vectors:", index.ntotal)

Stored Vectors: 16


**Step 10 - Ask Question**

In [9]:
question = input("Ask a Question: ")

Ask a Question:  What is pandas?


**Step 11 - Retrieve Context**

In [10]:
query_embedding = embedding_model.encode(
    [question]
)

D, I = index.search(
    np.array(query_embedding).astype("float32"),
    k=3
)

context = ""

for idx in I[0]:
    context += chunks[idx]
    context += "\n\n"

print(context[:1500])

Complete Guide of Pandas : By Aniket Sharma 
 
1. What is Pandas in Python, and why is it used? 
Question: What is the primary purpose of the Pandas library in 
Python, and what are its core data structures? 
Answer: 
Pandas is an open-source Python library used for data manipulation 
and analysis. It provides easy-to-use data structures and tools for 
handling structured data, such as tabular data in spreadsheets or SQL 
tables. 
Core Data Structures: 
 Series: A one-dimensional labeled array 

capable of holding any 
data type. 
 DataFrame: A two-dimensional labeled data structure with 
rows and columns, similar to a spreadsheet or SQL table. 
Pandas is widely used for tasks like data cleaning, transformation, 
aggregation, and merging. 2. Explain the difference between a Pandas Series and 
a DataFrame. 
Question: How do Series and DataFrames differ in Pandas? 
Series: 
 One-dimensional labeled array. 
 Homogeneous data (all elements are of the same type). 
 Example: A single co

**Step 12 - Generate Answer**

In [29]:
response = client.models.generate_content(
    model="models/gemini-flash-latest",
    contents=f"""
Question:
{question}

Context:
{context[:2000]}

Answer:
"""
)

print(response.text)

In the context of Python (the programming language Pandas is built on), the term **"exit"** refers to terminating a Python shell, Jupyter Notebook, or running script. 

Here is how you can exit Python:

### 1. In the Python Interactive Shell / Terminal:
To exit an active Python session, you can use one of the following methods:
* Run the built-in function: 
  ```python
  exit()
  ```
  or 
  ```python
  quit()
  ```
* **Keyboard Shortcuts:**
  * **Linux/macOS:** Press `Ctrl + D`
  * **Windows:** Press `Ctrl + Z` and then hit `Enter`

### 2. In a Python Script:
To stop the execution of a Python script cleanly at a specific point, you can use the `sys` module:
```python
import sys
sys.exit()
```


**Step 13 - Continuous Chat**

In [27]:
while True:

    question = input("\nAsk Question (type exit to stop): ")

    if question.lower() == "exit":
        break

    query_embedding = embedding_model.encode([question])

    D, I = index.search(
        np.array(query_embedding).astype("float32"),
        k=3
    )

    context = ""

    for idx in I[0]:
        context += chunks[idx]
        context += "\n\n"

    response = client.models.generate_content(
    model="models/gemini-flash-latest",
    contents=f"""
Question:
Question:
{question}

Context:
{context[:2000]}

Answer:
"""
)

    print(response.text)

    print("\nAnswer:")
    print(response.text)


Ask Question (type exit to stop):  Why we are using pandas?


Based on the provided context, we use Pandas for **data manipulation and analysis**. 

It is designed to provide easy-to-use data structures and tools for handling structured data, such as tabular data in spreadsheets or SQL tables.

Answer:
Based on the provided context, we use Pandas for **data manipulation and analysis**. 

It is designed to provide easy-to-use data structures and tools for handling structured data, such as tabular data in spreadsheets or SQL tables.



Ask Question (type exit to stop):  exit
